In [2]:
import pandas as pd

#zip file processing
import zipfile

#directory controls
import os

import io

In [4]:
#change to your file directory
ERC_DATA = "..\\datasets\\ERC20-stablecoins.zip"
GFC_DATA="..\\datasets\\gfc.zip"

In [5]:
os.makedirs("../datasets/processed/erc_20", exist_ok=True)  # creates nested directories
os.makedirs("../datasets/processed/gfc_data", exist_ok=True)

In [6]:
def describe(f, name):
    """
    To understand more about the dataframe (columns, num of null values, shape)
    Args:
        f -> String representing : file path
        name -> String : file name
    Output:
        None
    """
    if name.startswith("event_data"):
        df = pd.read_csv(f, encoding='latin-1')
    else:
        df = pd.read_csv(f)
    print(f"\n📋 All columns in {name}")
    print(df.columns.tolist())
    print(f"\n📊 First 5 rows:")
    print(df.head(5))
    print(f"\n Dataframe shape")
    print(df.shape)
    print(f"\n Number of null values")
    print(df.isna().sum())
    return df

# Processing ERC 20 Data

In [ ]:
# data_dataframes = {}
# with zipfile.ZipFile(ERC_DATA) as z:
#     for name in z.namelist():

#         if name.endswith(".zip"):
#             # if it is another zipped folder
#             print(f"Opening nested zip {name}")
#             nested_zip = z.read(name)

#             with zipfile.ZipFile(io.BytesIO(nested_zip)) as nested_z:
#                 for fileName in nested_z.namelist():
#                     with nested_z.open(fileName) as nested_f:
#                         df = pd.read_csv(nested_f)
#                         df['date'] = pd.to_datetime(df["timestamp"], unit="s") #convert unix timestamp to date time
#                         df["coins"] = fileName.split("/")[1].split("_")[0] #get the coin name as column
#                         df.to_csv(f"../datasets/processed/erc_20/processed_{fileName.split("/")[1]}", index=False)
#         else:
#             if ("token" in name):
#                 continue

#             else:
#                 print(f"Opening file {name}")
#                 # if it is a file
#                 with z.open(name) as f:
#                     df = describe(f, name)

#                     if ("timestamp" in df.columns): #conditions according to column name
#                         df["date"] = pd.to_datetime(df["timestamp"], unit="s") #convert unix timestamp to date time
#                     elif ("time_stamp" in df.columns):
#                         df["date"] = pd.to_datetime(df["time_stamp"], unit="s") #convert unix timestamp to date time
#                     df.to_csv(f"../datasets/processed/erc_20/processed_{name}", index=False)

Opening nested zip price_data.zip
dai
Opening file event_data.csv

📋 All columns in event_data.csv
['event', 'timestamp', 'type', 'stablecoin']

📊 First 5 rows:
                                               event   timestamp      type  \
0  BlackRock and Fidelity Back USDC in $400 Milli...  1649721600  positive   
1  Terra UST takes over BUSD to become third larg...  1650412800  positive   
2  LARGE amounts of UST selling on ANCHOR (approx...  1651881600  negative   
3  UST depegs LFG deploys assets to defend peg (7...  1651968000  negative   
4    UST Depegs again to 35 cents LUNA keeps falling  1652054400  negative   

  stablecoin  
0       usdc  
1       ustc  
2       ustc  
3       ustc  
4       ustc  

 Dataframe shape
(38, 4)

 Number of null values
event         0
timestamp     0
type          0
stablecoin    0
dtype: int64


In [ ]:
def describe_and_clean(df, name):
    """Integrated: Profiling + Cleaning"""
    # 1. PROFILE (Partner 2's Logic)
    print(f"\n📋 Profiling {name} | Shape: {df.shape} | Nulls: {df.isna().sum().sum()} | Duplicates: {df.duplicated().sum()}")

    # 2. STANDARDIZE TIMESTAMPS (Partner 2's Logic)
    for col in ["timestamp", "time_stamp"]:
        if col in df.columns:
            df["date"] = pd.to_datetime(df[col], unit="s", utc=True)

    return df

with zipfile.ZipFile(ERC_DATA, "r") as z:
    for name in z.namelist():
        # Handle Nested Zips
        if name.endswith(".zip"):
            print(f"\n📦 Processing Nested Zip: {name}")
            nested_zip_data = z.read(name)
            with zipfile.ZipFile(io.BytesIO(nested_zip_data)) as nested_z:
                for inner_name in nested_z.namelist():
                    if inner_name.endswith(".csv"):
                        with nested_z.open(inner_name) as f:
                            df = pd.read_csv(f)
                            # Apply the integrated logic
                            df = describe_and_clean(df, inner_name)
                            # Add coin label from filename
                            df["coins"] = inner_name.split("/")[-1].split("_")[0].upper()
                            df = df.drop_duplicates()  # <--- Add it here

                            # Save with the 'processed_' prefix
                            save_name = f"processed_{inner_name.split('/')[-1]}"
                            df.to_csv(os.path.join("../datasets/processed/erc_20", save_name), index=False)

       # Handle regular files
        elif name.endswith(".csv"):
            with z.open(name) as f:
                encoding = 'latin-1' if name == "event_data.csv" else None
                df = pd.read_csv(f, encoding=encoding)

                # 1. Profile (This shows the 998 duplicates)
                df = describe_and_clean(df, name)

                # 2. THE FIX: Actually remove the duplicates
                df = df.drop_duplicates()
                print(f"   ✅ Cleaned: {len(df)} unique rows remaining.")

                # 3. Save
                df.to_csv(os.path.join("../datasets/processed/erc_20", f"processed_{name}"), index=False)


📋 Profiling token_transfers.csv | Shape: (5280131, 7) | Nulls: 0 | Duplicates: 998
   ✅ Cleaned: 5279133 unique rows remaining.

📋 Profiling token_transfers_V2.0.0.csv | Shape: (28674511, 7) | Nulls: 0 | Duplicates: 10779
   ✅ Cleaned: 28663732 unique rows remaining.

📋 Profiling token_transfers_V3.0.0.csv | Shape: (36723655, 7) | Nulls: 0 | Duplicates: 12521
   ✅ Cleaned: 36711134 unique rows remaining.
📦 Processing Nested Zip: price_data.zip

📋 Profiling price_data/dai_price_data.csv | Shape: (215, 5) | Nulls: 0 | Duplicates: 0

📋 Profiling price_data/pax_price_data.csv | Shape: (215, 5) | Nulls: 0 | Duplicates: 0

📋 Profiling price_data/usdc_price_data.csv | Shape: (215, 5) | Nulls: 0 | Duplicates: 0

📋 Profiling price_data/usdt_price_data.csv | Shape: (215, 5) | Nulls: 0 | Duplicates: 0

📋 Profiling price_data/ustc_price_data.csv | Shape: (215, 5) | Nulls: 0 | Duplicates: 0

📋 Profiling price_data/wluna_price_data.csv | Shape: (215, 5) | Nulls: 0 | Duplicates: 0

📋 Profiling event

# Processing GFC Data

In [ ]:
# for filename in os.listdir(GFC_DATA_FOLDER):
#     if filename.endswith(".csv"):
#         file_path = os.path.join(GFC_DATA_FOLDER, filename)
#         if os.path.isfile(file_path):
#             print(filename)

#             df = describe(file_path, filename) #understand the file
#             print(f"Processing file for {df.iloc[0,1]}")
#             #formating df's format
#             ticker = df.iloc[0, 1] #get ticker value
#             date= df.iloc[2:, 0] # get date column
#             df = df.iloc[3:,:] # get the relevant dataset (from row 2 onwards)
#             df['ticker'] = ticker #assigned ticker to ticker column
#             df['date'] = date #assigned date to date column
#             df.to_csv(f"../datasets/processed/gfc_data/processed_{filename}", index=False)


In [5]:
# format detection
def is_yahoo_style(df):
    """Detect Yahoo multi-row header format."""
    return (
        "Price" in df.columns and
        len(df) >= 2 and
        str(df.iloc[0, 0]) == "Ticker" and
        str(df.iloc[1, 0]) == "Date"
    )


# cleaners
def yahoo_clean(df, filename):
    """Apply Yahoo metadata cleanup."""
    print(f"✨ Yahoo-cleaning → {filename}")

    ticker_val = df.iloc[0, 1]

    df = df.iloc[2:].copy()
    df.rename(columns={"Price": "Date"}, inplace=True)

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["ticker"] = ticker_val

    return df


def apply_unified_format(df, filename):
    """Apply shared formatting rules to all files."""
    print(f"📊 Formatting → {filename}")

    # Ensure ticker exists
    if "ticker" not in df.columns:
        if len(df) > 0 and df.shape[1] > 1:
            df["ticker"] = df.iloc[0, 1]
        else:
            df["ticker"] = "UNKNOWN"

    # Standardize date column
    if "Date" in df.columns:
        df["date_standardized"] = pd.to_datetime(df["Date"], errors="coerce")
    else:
        first_col = df.columns[0]
        df["date_standardized"] = pd.to_datetime(df[first_col], errors="coerce")

    df.reset_index(drop=True, inplace=True)

    return df


# file processor
def process_and_save(file_handle, filename, output_folder):

    clean_name = os.path.basename(filename)
    print(f"\n📄 Processing {clean_name}")

    try:
        df = pd.read_csv(file_handle)

        # Step 1 — Yahoo cleanup (if applicable)
        if is_yahoo_style(df):
            df = yahoo_clean(df, clean_name)

        # Step 2 — Unified formatting (always applied)
        df = apply_unified_format(df, clean_name)

        # Save
        save_path = os.path.join(output_folder, f"processed_{clean_name}")
        df.to_csv(save_path, index=False)

        print(f"✅ Saved → {save_path}")

    except Exception as e:
        print(f"❌ Failed → {clean_name}: {e}")


# pipeline runner
def run_unified_pipeline(source_path, output_folder):

    os.makedirs(output_folder, exist_ok=True)

    # ZIP source
    if zipfile.is_zipfile(source_path):
        print(f"📦 ZIP detected: {source_path}")

        with zipfile.ZipFile(source_path, "r") as z:
            for name in z.namelist():
                if name.endswith(".csv"):
                    with z.open(name) as f:
                        process_and_save(f, name, output_folder)

    # Folder source
    elif os.path.isdir(source_path):
        print(f"📂 Folder detected: {source_path}")

        for root, _, files in os.walk(source_path):
            for name in files:
                if name.endswith(".csv"):
                    path = os.path.join(root, name)
                    with open(path, "rb") as f:
                        process_and_save(f, name, output_folder)

    else:
        print("❌ Invalid source path")


# run this line
run_unified_pipeline(GFC_DATA, "../datasets/processed/gfc_data/")

📦 ZIP detected: ..\datasets\gfc.zip

📄 Processing AIG.csv
✨ Yahoo-cleaning → AIG.csv
📊 Formatting → AIG.csv
✅ Saved → ../datasets/processed/gfc_data/processed_AIG.csv

📄 Processing ^VIX.csv
✨ Yahoo-cleaning → ^VIX.csv
📊 Formatting → ^VIX.csv
✅ Saved → ../datasets/processed/gfc_data/processed_^VIX.csv

📄 Processing C.csv
✨ Yahoo-cleaning → C.csv
📊 Formatting → C.csv
✅ Saved → ../datasets/processed/gfc_data/processed_C.csv

📄 Processing JPM.csv
✨ Yahoo-cleaning → JPM.csv
📊 Formatting → JPM.csv
✅ Saved → ../datasets/processed/gfc_data/processed_JPM.csv

📄 Processing ^GSPC.csv
✨ Yahoo-cleaning → ^GSPC.csv
📊 Formatting → ^GSPC.csv
✅ Saved → ../datasets/processed/gfc_data/processed_^GSPC.csv

📄 Processing WGS3MO.csv
📊 Formatting → WGS3MO.csv
✅ Saved → ../datasets/processed/gfc_data/processed_WGS3MO.csv

📄 Processing ^DJI.csv
✨ Yahoo-cleaning → ^DJI.csv
📊 Formatting → ^DJI.csv
❌ Failed → ^DJI.csv: [Errno 13] Permission denied: '../datasets/processed/gfc_data/processed_^DJI.csv'

📄 Processing 